# Table 3 — Three-Model Inventory ↔ Susceptibility Linkage (Frequency Ratio + Hit Rate)

Reproduces the manuscript **Table 3** across the three models — **A: Data-Driven (DD)**,
**B: PINN-SMOTE**, **C: PINN-DCE** — over the full North Cotabato province set, linking
each model's susceptibility output to the documented landslide inventory.

### Definitions (count-based, quantile tertiles)
- **Classes**: Low / Moderate / High = **equal-count quantile tertiles** of each model's own
  susceptibility (each class ≈ ⅓ of the province slope units). Tertiles are assigned by
  **rank** (ties broken by order) so they stay equal-count even when a model's output is
  saturated/tied — value-edge quantiles collapse on such outputs.
- **Distribution (Low/Moderate/High %)**: share of the inventory (`landslide==1` units)
  falling in each class (sums to 100%).
- **Hit rate (High)** = % of the inventory in the High class.
- **Frequency Ratio FR(High)** = (% inventory in High) ÷ (% units in High = n_High / N_total).
  FR > 1 ⇒ landslides over-represented in High.

### Caveats (read before citing)
- **FR ≈ 3 × hit-rate** here: with equal-count tertiles n_High = N/3 for every model, so the
  High-class denominator is constant and FR is determined by the hit rate. (The draft's
  *varying* High counts came from fixed thresholds; quantile tertiles replace that.)
- The PINN outputs are **saturated** (a large mass sits at the floor), so the Low/Moderate
  boundary falls *within* a single tied value — the Low vs Moderate split is partly
  arbitrary. The meaningful contrast is **High vs the rest**.
- The models were trained on only 3 soil types (`Undifferentiated`, `Sandy Clay Loam`,
  `Loam`); other province soil classes hit the `[UNK]`/out-of-distribution path.
- Province set is **Parameters v2 (386,283 raw)**; the notebook reports the **actual**
  post-filter N and inventory count.

See `docs/frequency_ratio_classing.md` for the full method and references.

In [1]:
import os, sys
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
# project root for `py_files` package; py_files/ itself so bare module names
# (e.g. `Landslidev2`) resolve when Keras deserializes the saved DD model.
for p in (str(PROJECT_ROOT), str(PROJECT_ROOT / "py_files")):
    if p not in sys.path:
        sys.path.insert(0, p)

import importlib
import numpy as np
import pandas as pd
import pyogrio
import tensorflow as tf
from tensorflow.keras.models import load_model

# NOTE: import the submodules via import_module. `from py_files import metrics`
# returns keras.metrics here because the package __init__ shadows that name.
metrics = importlib.import_module("py_files.metrics")
GallenModel_v1 = importlib.import_module("py_files.GallenModel_v1")
dataframe_to_dataset = importlib.import_module("py_files.data").dataframe_to_dataset

DATA = "/Users/giogonzales/Documents/ml-prep/ML-PREP-2025/learn/data"
TW = "/Users/giogonzales/Documents/ml-prep/ML-PREP-2025/learn/trainedWeights"
PROVINCE = f"{DATA}/North Cotabato Slope Unit - Parameters v2.gpkg" 

## 1. Load & preprocess the province set
Filter `Slope_mean ≥ 10`, drop NaNs, rename `Landslide`→`landslide`, reconstruct the models'
categorical inputs from the string `Soil Type`. Count-based FR needs no geometry, so we read
without it (much lighter than the 689 MB file with geometries).

In [2]:
NUMERIC18 = ['Clay_mean','Sand_mean','Silt_mean','NDVI_mean','Est_mean','Nrt_mean',
             'HorCurv_mean','VertCurv_mean','Slope_mean','Elev_mean','SoilThc_mean',
             'DistFlt_min','LULC_majority','TWI_mean','Prc_mean','Distrv_min',
             'distrd_min','BUK_mean']
# string soil class -> int code, from the SU_15 training encoding (Hydrosol unseen -> -1).
SOIL_MAP = {'Clay':2,'Clay Loam':4,'Loam':6,'Sand':3,'Sandy Clay Loam':1,
            'Silty Clay Loam':5,'Undifferentiated':0}

cols = list(dict.fromkeys(NUMERIC18 + ['PGA1_max','Soil Type','Slope_mean','Landslide']))
df = pyogrio.read_dataframe(PROVINCE, columns=cols, read_geometry=False)
print("raw province rows:", len(df))

df = df[df['Slope_mean'] >= 10]
df = df.dropna(subset=[c for c in cols if c != 'Landslide'])
df = df.rename(columns={'Landslide': 'landslide'})
df['landslide'] = df['landslide'].astype(int)
df['type'] = df['Soil Type'].astype(str)                               # DCE: string input
df['Soil Type'] = df['type'].map(SOIL_MAP).fillna(-1).astype(float)    # SMOTE/DD: numeric code
df['Soil_Type'] = df['Soil Type']                                      # DD alias
df = df.reset_index(drop=True)

N_TOTAL = len(df)
N_INV = int((df['landslide'] == 1).sum())
print(f"after filter: N_total={N_TOTAL:,}   inventory(landslide==1)={N_INV:,}")
print("\nsoil class counts:\n", df['type'].value_counts().to_string())

raw province rows: 386283
after filter: N_total=255,276   inventory(landslide==1)=5,961

soil class counts:
 type
Undifferentiated    145188
Clay                 60155
Sandy Clay Loam      27552
Clay Loam            20803
Loam                  1430
Silty Clay Loam         85
Sand                    63


## 2. Load the three models and predict
Each model has its own input contract:
- **DD** (`model2.keras`) — subclassed `CotabatoModel`; manual tensor dict; `Soil_Type` int code.
- **PINN-SMOTE** (`v3/fold-10`) — physics PINN; `Soil Type` numeric; `PGA1_max`.
- **PINN-DCE** (`v4-1-1dce/fold-4`) — physics PINN; `type` string; `PGA1_max`.

`GallenModel.py` and `GallenModel_v1.py` register layers under the same names, so we pass
**explicit `custom_objects`** (pinned to `GallenModel_v1`) to the PINN loads — this overrides
the global registry regardless of import order.

In [3]:
PINN_OBJECTS = {n: getattr(GallenModel_v1, n) for n in (
    'CohesionLayer','InternalFrictionLayer','DisplacementLayer',
    'NewmarkActivation','LandslideActivationLayer') if hasattr(GallenModel_v1, n)}

# --- PINN-SMOTE (v3/fold-10). NOTE: the v2-2 SMOTE-training notebook uses v3-1FOS/fold-10;
#     swap the path if you want that variant. v3/ matches the existing "PINN SMOTE" figure. ---
smote = load_model(f"{TW}/trainedCotabatoPhase7/historical/v3/fold-10-model-0.keras",
                   custom_objects=PINN_OBJECTS, compile=False)
SMOTE_COLS = NUMERIC18 + ['Soil Type', 'PGA1_max', 'landslide']
preds_smote = np.asarray(
    smote.predict(dataframe_to_dataset(df[SMOTE_COLS].copy(), shuffle=False, batch_size=4096),
                  verbose=0)).flatten()

# --- PINN-DCE (v4-1-1dce/fold-4) ---
dce = load_model(f"{TW}/trainedCotabatoPhase7/historical/v4-1-1dce/fold-4-model-0.keras",
                 custom_objects=PINN_OBJECTS, compile=False)
DCE_COLS = NUMERIC18 + ['type', 'PGA1_max', 'landslide']
preds_dce = np.asarray(
    dce.predict(dataframe_to_dataset(df[DCE_COLS].copy(), shuffle=False, batch_size=4096),
                verbose=0)).flatten()

# --- DD (model2.keras). Import Landslidev2 to register CotabatoModel for deserialization. ---
import Landslidev2  # noqa: F401
dd = load_model(f"{TW}/data-driven/model2.keras", compile=False)
DD_FEATURES = dd.get_config()['features']
dd_input = {c: tf.convert_to_tensor(df[c].values.reshape(-1, 1), dtype=tf.float32)
            for c in DD_FEATURES}
preds_dd = np.asarray(dd.predict(dd_input, verbose=0)).flatten()

for nm, p in [("DD", preds_dd), ("PINN-SMOTE", preds_smote), ("PINN-DCE", preds_dce)]:
    print(f"{nm:11s} preds: n={len(p):,} min={p.min():.4f} max={p.max():.4f} "
          f"mean={p.mean():.4f} n_unique={np.unique(p).size:,}")

DD          preds: n=255,276 min=0.0512 max=0.9981 mean=0.1096 n_unique=72,525
PINN-SMOTE  preds: n=255,276 min=0.1326 max=1.0000 mean=0.2945 n_unique=49,247
PINN-DCE    preds: n=255,276 min=0.1326 max=1.0000 mean=0.3837 n_unique=76,313


## Binary landslide rasters (threshold = 0.5)

Single-band GeoTIFFs where each slope unit is **1 = landslide** / **0 = no landslide** (susceptibility >= 0.5), for the three models DCE, SMOTE, DD. Written to `figures/`.

In [ ]:
# --- Binary landslide / no-landslide rasters (threshold = 0.5) for all 3 models ---
# `df` was read WITHOUT geometry (cell above), so re-read the province polygons with
# the IDENTICAL filter (Slope>=10, dropna on the same columns) so the geometry aligns
# 1:1 with df rows -- and therefore with preds_dce / preds_smote / preds_dd.
import geopandas as gpd
THRESH = 0.5

_geom = gpd.read_file(PROVINCE)
gdf_geom = (
    _geom[_geom['Slope_mean'] >= 10]
    .dropna(subset=[c for c in cols if c != 'Landslide'])
    .reset_index(drop=True)
)
assert len(gdf_geom) == len(df), f"geometry rows {len(gdf_geom)} != df rows {len(df)}"

FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

for nm, p, fname in [
    ("PINN-DCE",   preds_dce,   "dce_binary_landslide_thresh0.5.tif"),
    ("PINN-SMOTE", preds_smote, "smote_binary_landslide_thresh0.5.tif"),
    ("DD",         preds_dd,    "dd_binary_landslide_thresh0.5.tif"),
]:
    binary = (p >= THRESH).astype("float64")   # 1 = landslide, 0 = no landslide
    print(f"{nm:11s} landslide pixels (p>={THRESH}): {int(binary.sum()):,} / {len(binary):,}")
    metrics.rasterize_to_geotiff(
        gdf_geom, binary,
        FIG_DIR / fname,
        pixel_size=30.0,
        layer_name=f"{nm}_landslide_ge_{THRESH}",
    )


## 3. Per-model Frequency-Ratio table (3-class quantile tertiles, count-based)
`scheme="quantile", area_based=False, n_classes=3` ⇒ rank-based equal-count tertiles.

In [4]:
LBL = ["Low", "Moderate", "High"]
results = {}
for nm, p in [("DD", preds_dd), ("PINN-SMOTE", preds_smote), ("PINN-DCE", preds_dce)]:
    print(f"\n===== {nm} =====")
    results[nm] = metrics.frequency_ratio_table(
        df, p, label_col="landslide", model_name=nm,
        scheme="quantile", area_based=False, n_classes=3, labels=LBL)


===== DD =====
[DD] scheme='quantile'  classes=3  edges: rank-tertiles
model    class      sus_range  n_slope_units  area_km2  pct_area  n_landslides  pct_landslides  freq_ratio  cum_pct_area  cum_hit_rate
   DD     High [0.052, 0.998]          85092     0.085     0.333          5856           0.982       2.947         0.333         0.982
   DD Moderate [0.052, 0.052]          85092     0.085     0.333            45           0.008       0.023         0.667         0.990
   DD      Low [0.051, 0.052]          85092     0.085     0.333            60           0.010       0.030         1.000         1.000

===== PINN-SMOTE =====
[PINN-SMOTE] scheme='quantile'  classes=3  edges: rank-tertiles
     model    class      sus_range  n_slope_units  area_km2  pct_area  n_landslides  pct_landslides  freq_ratio  cum_pct_area  cum_hit_rate
PINN-SMOTE     High [0.133, 1.000]          85092     0.085     0.333          4907           0.823       2.470         0.333         0.823
PINN-SMOTE Moderate 

## 4. Compact Table 3 (one row per model) + save

In [5]:
table3 = metrics.frequency_ratio_summary(results)

out = PROJECT_ROOT / "figures" / "table3_three_model_fr.csv"
out.parent.mkdir(parents=True, exist_ok=True)
table3.to_csv(out, index=False)
print(f"\nSaved -> {out}")
print(f"Province N_total = {N_TOTAL:,}   inventory = {N_INV:,}   (draft cited ~481,000 / 3,140)")
table3

     model  High (%)  Moderate (%)  Low (%)  hit_rate_high  fr_high  n_high  n_total
        DD     98.24          0.75     1.01          98.24     2.95   85092   255276
PINN-SMOTE     82.32          0.00    17.68          82.32     2.47   85092   255276
  PINN-DCE     70.94         10.07    18.99          70.94     2.13   85092   255276

Saved -> /Users/giogonzales/Documents/ml-prep/mlprep/figures/table3_three_model_fr.csv
Province N_total = 255,276   inventory = 5,961   (draft cited ~481,000 / 3,140)


,model,High (%),Moderate (%),Low (%),hit_rate_high,fr_high,n_high,n_total
0,DD,98.238551,0.754907,1.006543,98.238551,2.947157,85092,255276
1,PINN-SMOTE,82.318403,0.000000,17.681597,82.318403,2.469552,85092,255276
2,PINN-DCE,70.944472,10.065425,18.990102,70.944472,2.128334,85092,255276


## Notes for the manuscript
- **N_total** and **inventory** above are the actual post-filter counts on Parameters v2;
  reconcile the caption's ~481,000 / 3,140 against them.
- With quantile tertiles, **n_High = N/3 for all models**, so FR(High) = hit_rate / (1/3)
  ≈ 3 × hit-rate fraction — report hit rate and FR together but note they are not independent.
- Soil types outside the 3-type training vocabulary are out-of-distribution; quantify the
  affected fraction from the soil-class counts in §1 if reviewers ask.
- The Low/Moderate split for the PINNs falls inside a tied (saturated) value and is partly
  arbitrary; lead the narrative with **High vs rest**.